# Assignment 3.1 — Neighborhood Feature Group
**Mostafa Zamaniturk**

Build a SageMaker Feature Store **Neighborhood** feature group from `housing.csv` and `housing_gmaps_data_raw.csv`, then query Brooktree, Fisherman's Wharf, and Los Osos.

**Kernel:** Python 3 (Data Science) in SageMaker Studio (or equivalent with `boto3` / `sagemaker`).

## 1. Setup SageMaker Feature Store

In [ ]:
%pip install 'boto3>1.17.21' 'sagemaker<3.0' pandas numpy -q

In [ ]:
import boto3
import sagemaker

print("boto3:", boto3.__version__)
print("sagemaker:", sagemaker.__version__)

In [ ]:
from sagemaker.session import Session
from sagemaker import get_execution_role

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-housing-neighborhood"
role = get_execution_role()

print("region:", region)
print("bucket:", default_s3_bucket_name)
print("role:", role)

## 2. Load and join datasets

Join housing features to Google Maps metadata on `longitude` / `latitude`. Keep rows that have a `neighborhood-political` value (this becomes the Feature Group primary key).

In [ ]:
import math
import time
from time import gmtime, strftime, sleep

import numpy as np
import pandas as pd

housing = pd.read_csv("housing.csv")
gmaps = pd.read_csv("housing_gmaps_data_raw.csv")

print("housing:", housing.shape)
print("gmaps:", gmaps.shape)
print("ocean_proximity values:\n", housing["ocean_proximity"].value_counts())

gmaps_keys = gmaps.drop_duplicates(subset=["longitude", "latitude"])
df = housing.merge(gmaps_keys, on=["longitude", "latitude"], how="inner")

# Primary key comes from neighborhood-political
df = df[df["neighborhood-political"].notna()].copy()
df = df[df["neighborhood-political"].astype(str).str.strip() != ""].copy()
df["neighborhood"] = df["neighborhood-political"].astype(str).str.strip()

print("rows with neighborhood:", df.shape[0])
print("unique neighborhoods:", df["neighborhood"].nunique())
df.head()

## 3. Feature engineering (Neighborhood Feature Group)

| Feature | Rule |
|---|---|
| `neighborhood` | primary key from `neighborhood-political` |
| `event_time` | Unix time at ingestion |
| `lt_1h_ocean`, `inland`, `island`, `near_bay`, `near_ocean` | one-hot from `ocean_proximity` (mode per neighborhood) |
| `median_house_value` | mean per neighborhood, capped at 500,000 |
| `median_house_age` | mean of `housing_median_age`, discretized into 10-year bins (`0-9`, `10-19`, ...) |
| `total_households` | mean of `households`, rounded **up** to integer |
| `bedrooms_per_household` | `total_bedrooms / households`; impute missing bedrooms with postal-code average |

Feature Store names cannot contain `<` or spaces, so `<1H OCEAN` → `lt_1h_ocean`, `NEAR BAY` → `near_bay`, etc.

In [ ]:
# Impute missing total_bedrooms using postal-code average, then global mean as fallback
df["postal_code"] = df["postal_code"].astype(str)
postal_mean_bedrooms = df.groupby("postal_code")["total_bedrooms"].transform("mean")
df["total_bedrooms_imputed"] = df["total_bedrooms"].fillna(postal_mean_bedrooms)
df["total_bedrooms_imputed"] = df["total_bedrooms_imputed"].fillna(df["total_bedrooms"].mean())

df["bedrooms_per_household"] = df["total_bedrooms_imputed"] / df["households"]

# Aggregate to neighborhood level
neighborhood_df = (
    df.groupby("neighborhood", as_index=False)
    .agg(
        median_house_value=("median_house_value", "mean"),
        median_house_age=("housing_median_age", "mean"),
        total_households=("households", "mean"),
        bedrooms_per_household=("bedrooms_per_household", "mean"),
        ocean_proximity=(
            "ocean_proximity",
            lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0],
        ),
    )
)

# Cap median house value at 500,000
neighborhood_df["median_house_value"] = neighborhood_df["median_house_value"].clip(upper=500000.0)

# Discretize average house age into 10-year bins: 0-9, 10-19, ...
def age_bin(age: float) -> str:
    lo = int(age) // 10 * 10
    return f"{lo}-{lo + 9}"


neighborhood_df["median_house_age"] = neighborhood_df["median_house_age"].apply(age_bin)

# Round average households up to an integer
neighborhood_df["total_households"] = np.ceil(neighborhood_df["total_households"]).astype(int)

# One-hot encode ocean_proximity (ensure all expected categories exist)
ocean_categories = ["<1H OCEAN", "INLAND", "ISLAND", "NEAR BAY", "NEAR OCEAN"]
ocean_dummies = pd.get_dummies(neighborhood_df["ocean_proximity"])
for cat in ocean_categories:
    if cat not in ocean_dummies.columns:
        ocean_dummies[cat] = 0
ocean_dummies = ocean_dummies[ocean_categories].astype(int)

ocean_rename = {
    "<1H OCEAN": "lt_1h_ocean",
    "INLAND": "inland",
    "ISLAND": "island",
    "NEAR BAY": "near_bay",
    "NEAR OCEAN": "near_ocean",
}
ocean_dummies = ocean_dummies.rename(columns=ocean_rename)

neighborhood_df = pd.concat([neighborhood_df.drop(columns=["ocean_proximity"]), ocean_dummies], axis=1)

# Event time = ingestion timestamp (seconds since epoch)
current_time_sec = int(round(time.time()))
neighborhood_df["event_time"] = pd.Series(
    [float(current_time_sec)] * len(neighborhood_df), dtype="float64"
)

# Column order for Feature Group
feature_columns = [
    "neighborhood",
    "event_time",
    "lt_1h_ocean",
    "inland",
    "island",
    "near_bay",
    "near_ocean",
    "median_house_value",
    "median_house_age",
    "total_households",
    "bedrooms_per_household",
]
neighborhood_df = neighborhood_df[feature_columns]

print("neighborhood feature rows:", neighborhood_df.shape)
neighborhood_df.head()

In [ ]:
# Quick local preview of the three required neighborhoods (before Feature Store query)
preview_names = ["Brooktree", "Fisherman's Wharf", "Los Osos"]
neighborhood_df[neighborhood_df["neighborhood"].isin(preview_names)]

## 4. Create Neighborhood Feature Group

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup

neighborhood_feature_group_name = "neighborhood-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

neighborhood_feature_group = FeatureGroup(
    name=neighborhood_feature_group_name,
    sagemaker_session=feature_store_session,
)

def cast_object_to_string(data_frame: pd.DataFrame) -> None:
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")


cast_object_to_string(neighborhood_df)

record_identifier_feature_name = "neighborhood"
event_time_feature_name = "event_time"

neighborhood_feature_group.load_feature_definitions(data_frame=neighborhood_df)
print("Feature group name:", neighborhood_feature_group_name)
neighborhood_feature_group.feature_definitions

In [ ]:
def wait_for_feature_group_creation_complete(feature_group: FeatureGroup) -> None:
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


neighborhood_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)

wait_for_feature_group_creation_complete(feature_group=neighborhood_feature_group)
neighborhood_feature_group.describe()

## 5. Ingest records into Feature Store

In [ ]:
neighborhood_feature_group.ingest(data_frame=neighborhood_df, max_workers=3, wait=True)
print(f"Ingested {len(neighborhood_df)} neighborhood records.")

## 6. Query Feature Values (for screenshots)

Query the online Feature Store for:

1. Brooktree  
2. Fisherman's Wharf  
3. Los Osos

In [ ]:
def get_neighborhood_record(neighborhood_name: str) -> dict:
    response = featurestore_runtime.get_record(
        FeatureGroupName=neighborhood_feature_group_name,
        RecordIdentifierValueAsString=neighborhood_name,
    )
    record = {item["FeatureName"]: item["ValueAsString"] for item in response.get("Record", [])}
    return record


query_neighborhoods = ["Brooktree", "Fisherman's Wharf", "Los Osos"]

for name in query_neighborhoods:
    print("=" * 60)
    print(f"Feature Store record: {name}")
    print("=" * 60)
    record = get_neighborhood_record(name)
    if not record:
        print("No record found.")
    else:
        for k, v in record.items():
            print(f"{k}: {v}")
    print()

In [ ]:
# Same three neighborhoods in one BatchGetRecord call (useful extra screenshot)
batch_response = featurestore_runtime.batch_get_record(
    Identifiers=[
        {
            "FeatureGroupName": neighborhood_feature_group_name,
            "RecordIdentifiersValueAsString": query_neighborhoods,
        }
    ]
)
batch_response

### Optional: Athena / Offline Store query

Online `get_record` is enough for the graded screenshots. If you also want offline-store SQL, wait a few minutes after ingestion, then run the cell below.

In [ ]:
neighborhood_query = neighborhood_feature_group.athena_query()
neighborhood_table = neighborhood_query.table_name

query_string = f"""
SELECT *
FROM "{neighborhood_table}"
WHERE neighborhood IN ('Brooktree', 'Fisherman''s Wharf', 'Los Osos')
"""
print("Running:", query_string)

neighborhood_query.run(
    query_string=query_string,
    output_location=f"s3://{default_s3_bucket_name}/{prefix}/query_results/",
)
neighborhood_query.wait()
athena_df = neighborhood_query.as_dataframe()
athena_df

## 7. Cleanup (optional)

Delete the Feature Group when you are done to avoid ongoing cost.

In [ ]:
# Uncomment when finished with screenshots / submission
# neighborhood_feature_group.delete()
# print("Deleted:", neighborhood_feature_group_name)